# Notebook 08: Monotonicity and Non-Crossing Quantiles

## Learning Objectives

By the end of this notebook, you will be able to:

1. Understand why quantile crossing violates theory
2. Detect crossing in estimated quantile curves
3. Apply rearrangement method (Chernozhukov et al. 2010)
4. Use isotonic regression for monotonicity
5. Implement constrained quantile regression
6. Compare methods and understand trade-offs (fit vs monotonicity)

## Duration
~150 minutes

## Prerequisites
- Notebooks 01–02 (Quantile Regression fundamentals, multiple quantiles)
- Optimization basics

## Dataset
Simulated data designed to exhibit quantile crossing (`crossing_example.csv`).

In [ ]:
# Standard libraries
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# Statistical libraries

# PanelBox imports
from panelbox.models.quantile import PooledQuantile
from panelbox.models.quantile.monotonicity import QuantileMonotonicity

# Visualization configuration
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)

# Reproducibility
np.random.seed(42)

# Define paths (relative to notebooks/)
BASE_DIR = Path("..")
OUTPUT_DIR = BASE_DIR / "outputs"
PLOTS_DIR = OUTPUT_DIR / "plots"
RESULTS_DIR = OUTPUT_DIR / "results"

# Create output directories
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete!")
print(f"Output directory: {OUTPUT_DIR}")

---

## 1. Introduction & Motivation

### Research Question

> *"What should we do when estimated quantiles cross?"*

Quantile regression estimates the conditional $\tau$-th quantile $Q_\tau(Y|X) = X'\beta(\tau)$. By definition, if $\tau_1 < \tau_2$, then:

$$Q_{\tau_1}(Y|X) \leq Q_{\tau_2}(Y|X) \quad \text{for all } X$$

However, because standard quantile regression estimates each quantile **independently**, the estimated curves $\hat{Q}_{\tau_1}(Y|X)$ and $\hat{Q}_{\tau_2}(Y|X)$ can **cross** for some values of $X$. This is known as the **quantile crossing problem**.

```
THEORETICAL REQUIREMENT:
  By definition: Q_τ₁(Y|X) ≤ Q_τ₂(Y|X) for all X when τ₁ < τ₂

EMPIRICAL REALITY:
  Estimated Q̂_τ₁(Y|X) may be > Q̂_τ₂(Y|X) for some X

CAUSES:
  1. Finite sample variation (noise)
  2. Model misspecification
  3. Insufficient sample at tails
  4. Covariate correlations

SOLUTIONS:
  1. Rearrangement (post-estimation correction)
  2. Isotonic regression (monotone smoothing)
  3. Constrained optimization (enforce during estimation)
  4. Location-scale model (prevents crossing by design)
```

---

## 2. Theoretical Concepts

### 2.1 Why Crossing is a Problem

**Logical Violation**: The $\tau$-th conditional quantile $Q_\tau(Y|X)$ is defined as the value such that:

$$P(Y \leq Q_\tau(Y|X) \mid X) = \tau$$

If $Q_{0.2}(Y|X) > Q_{0.8}(Y|X)$, this would mean "the 20th percentile is above the 80th percentile" — which is **impossible** by definition.

**Practical Implications of Crossing:**

- **Uninterpretable predictions**: Predicted quantile intervals are inverted
- **Negative conditional densities**: The implied density $f(y|X) = \partial \tau / \partial Q_\tau$ becomes negative
- **Violation of stochastic ordering**: The fundamental ordering of quantiles is broken
- **Publication/credibility issues**: Reviewers will rightfully question crossed quantiles

### 2.2 Rearrangement Method

**Idea**: Post-process estimates to restore monotonicity while minimally altering predictions.

**Algorithm** (Chernozhukov, Fernández-Val, Galichon 2010):

```
Input: Q̂_τ₁(X), ..., Q̂_τₖ(X) for τ₁ < ... < τₖ
Output: Q̃_τ₁(X), ..., Q̃_τₖ(X) (monotone)

For each X:
  1. Collect all predictions: {Q̂_τⱼ(X)}
  2. Sort them: Q̃_τ₁(X) ≤ Q̃_τ₂(X) ≤ ... ≤ Q̃_τₖ(X)
  3. This gives rearranged quantiles

Properties:
  ✓ Preserves marginal distributions
  ✓ Fast (just sorting)
  ✓ Post-estimation (no re-fitting)
  ✗ Can create "plateaus" (flat regions)
```

### 2.3 Isotonic Regression

**Idea**: Smooth quantile process to be monotone using the Pool-Adjacent-Violators (PAV) algorithm.

```
1. Scan predictions from left to right
2. When violation found (Q̂_τⱼ > Q̂_τⱼ₊₁):
   - Pool adjacent values
   - Replace with average
3. Repeat until monotone
```

**Properties**:
- Smooths out noise
- Can handle large violations
- More aggressive than rearrangement
- May alter predictions more

### 2.4 Constrained Optimization

**Idea**: Add monotonicity constraints during QR estimation.

**Formulation**:

$$\min_{\beta_1, \ldots, \beta_K} \sum_{j=1}^{K} \sum_{i=1}^{n} \rho_{\tau_j}(y_i - x_i'\beta_j)$$

subject to: $x_i'\beta_1 \leq x_i'\beta_2 \leq \ldots \leq x_i'\beta_K \quad \text{for all } i$

**Approaches**:
1. **Hard constraints**: Enforce $X\beta_{\tau_j} \leq X\beta_{\tau_{j+1}}$ exactly (trust-constr)
2. **Soft penalty**: $\lambda \sum \max(0, Q_{\tau_j} - Q_{\tau_{j+1}})^2$ (simultaneous QR)
3. **Projection**: Estimate unconstrained, then project to feasible set

---

## 3. Implementation

### 3.1 Load Data and Estimate Unconstrained QR

In [ ]:
# Load the simulated crossing example data
from panelbox.datasets import load_dataset

data = load_dataset("crossing_example", category="quantile")

print(f"Dataset shape: {data.shape}")
print(f"Variables: {data.columns.tolist()}")
print(f"N observations: {len(data)}")
print(f"N individuals: {data['id'].nunique()}")
print(f"N periods: {data['t'].nunique()}")
print()
print("Summary Statistics:")
display(data[["y", "x1", "x2"]].describe())
print()
print("First few rows:")
display(data.head(10))


In [ ]:
# Prepare arrays for quantile regression
y = data["y"].values
X = np.column_stack(
    [
        np.ones(len(data)),  # intercept
        data["x1"].values,
        data["x2"].values,
    ]
)
entity_id = data["id"].values
var_names = ["const", "x1", "x2"]

print(f"Dependent variable (y): {y.shape}")
print(f"Design matrix (X): {X.shape}")
print(f"Variables: {var_names}")
print(f"Unique entities: {len(np.unique(entity_id))}")

In [ ]:
# Estimate standard QR for multiple quantiles (unconstrained)
tau_grid = np.round(np.arange(0.1, 0.91, 0.1), 2)

qr_results = {}
print("Estimating unconstrained quantile regressions...")
for tau in tau_grid:
    model = PooledQuantile(y, X, entity_id=entity_id, quantiles=tau)
    qr_results[tau] = model.fit(se_type="cluster")
    print(f"  \u03c4 = {tau:.1f} done")

print(f"\nEstimated {len(tau_grid)} quantile models.")

# Display coefficients across quantiles
coef_table = pd.DataFrame(
    {
        "tau": tau_grid,
        "const": [qr_results[tau].params.ravel()[0] for tau in tau_grid],
        "x1": [qr_results[tau].params.ravel()[1] for tau in tau_grid],
        "x2": [qr_results[tau].params.ravel()[2] for tau in tau_grid],
    }
)
print("\nCoefficient Estimates by Quantile:")
display(coef_table)

### 3.2 Detect Crossing

The `QuantileMonotonicity.detect_crossing()` method checks whether the estimated quantile curves cross at any evaluation point.

In [ ]:
# Detect crossing using PanelBox
# We need to prepare results in the format expected by detect_crossing:
# {tau: result} where result.params is a 1D array and result.model.X exists

# Create compatible result objects for detect_crossing
class SimpleResult:
    """Lightweight result object compatible with QuantileMonotonicity."""

    def __init__(self, params, X):
        self.params = params
        self.model = type("Model", (), {"X": X, "nobs": len(X)})()


# Build results dict with 1D params
results_for_crossing = {}
for tau in tau_grid:
    params_1d = qr_results[tau].params.ravel()
    results_for_crossing[tau] = SimpleResult(params_1d, X)

# Detect crossing
crossing_report = QuantileMonotonicity.detect_crossing(results_for_crossing, X_test=X)

# Print report
crossing_report.summary()

# Also show as DataFrame if crossings exist
if crossing_report.has_crossing:
    print("\nCrossing Details:")
    display(crossing_report.to_dataframe())

### 3.3 Visualize Crossing

We visualize crossing by plotting the predicted quantile values for a representative observation. When quantiles cross, the curve is **non-monotone** — it goes down when it should only go up.

In [ ]:
# Compute predicted quantiles for ALL observations
predictions_all = np.column_stack([X @ results_for_crossing[tau].params for tau in tau_grid])

# Find observations with crossing
n_crossings_per_obs = np.zeros(len(X))
for i in range(len(tau_grid) - 1):
    n_crossings_per_obs += predictions_all[:, i] > predictions_all[:, i + 1] + 1e-10

obs_with_crossing = np.where(n_crossings_per_obs > 0)[0]
print(f"Observations with at least one crossing: {len(obs_with_crossing)} out of {len(X)}")
print(f"Percentage: {100 * len(obs_with_crossing) / len(X):.1f}%")

# Select a representative observation with crossing (or first obs if none)
if len(obs_with_crossing) > 0:
    # Pick the one with most crossings
    rep_idx = obs_with_crossing[np.argmax(n_crossings_per_obs[obs_with_crossing])]
else:
    rep_idx = 0

print(f"\nRepresentative observation index: {rep_idx}")
print(f"Covariates: x1={X[rep_idx, 1]:.3f}, x2={X[rep_idx, 2]:.3f}")
print(f"Number of crossings: {int(n_crossings_per_obs[rep_idx])}")

In [ ]:
# Predict for the representative observation (unconstrained)
x_rep = X[rep_idx : rep_idx + 1]  # (1, p)
preds_unconstrained = {}
for tau in tau_grid:
    preds_unconstrained[tau] = float(x_rep @ results_for_crossing[tau].params)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

pred_values = [preds_unconstrained[tau] for tau in tau_grid]
ax.plot(
    tau_grid,
    pred_values,
    "o-",
    linewidth=2,
    markersize=8,
    label="Unconstrained QR",
    color="steelblue",
)

# Mark crossings (where pred[i] > pred[i+1])
crossing_plotted = False
for i in range(len(tau_grid) - 1):
    if preds_unconstrained[tau_grid[i]] > preds_unconstrained[tau_grid[i + 1]]:
        label = "Crossing" if not crossing_plotted else ""
        ax.plot(
            [tau_grid[i], tau_grid[i + 1]],
            [preds_unconstrained[tau_grid[i]], preds_unconstrained[tau_grid[i + 1]]],
            "ro-",
            linewidth=4,
            markersize=10,
            label=label,
        )
        crossing_plotted = True

ax.set_xlabel("Quantile (\u03c4)", fontsize=12, fontweight="bold")
ax.set_ylabel("Predicted Value", fontsize=12, fontweight="bold")
ax.set_title("Quantile Crossing Visualization (Unconstrained)", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "08_crossing_unconstrained.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Plot saved to {PLOTS_DIR / '08_crossing_unconstrained.png'}")

In [ ]:
# Show multiple observation curves to see the pattern
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: Sample of quantile curves
n_show = min(30, len(X))
idx_show = np.random.choice(len(X), n_show, replace=False)

for i in idx_show:
    preds_i = [float(X[i : i + 1] @ results_for_crossing[tau].params) for tau in tau_grid]
    color = "red" if n_crossings_per_obs[i] > 0 else "steelblue"
    alpha = 0.7 if n_crossings_per_obs[i] > 0 else 0.3
    ax1.plot(tau_grid, preds_i, "-", alpha=alpha, color=color, linewidth=1.5)

# Add legend entries
ax1.plot([], [], "-", color="red", alpha=0.7, linewidth=1.5, label="With crossing")
ax1.plot([], [], "-", color="steelblue", alpha=0.3, linewidth=1.5, label="Monotone")
ax1.set_xlabel("Quantile (\u03c4)", fontsize=12, fontweight="bold")
ax1.set_ylabel("Predicted Value", fontsize=12, fontweight="bold")
ax1.set_title("Panel A: Sample of Quantile Curves", fontsize=14, fontweight="bold")
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)

# Panel B: Coefficient paths
for v_idx, (vname, color) in enumerate(zip(var_names, ["#0072B2", "#E69F00", "#009E73"])):
    coefs = [results_for_crossing[tau].params[v_idx] for tau in tau_grid]
    ax2.plot(tau_grid, coefs, "o-", linewidth=2, markersize=6, label=vname, color=color)

ax2.set_xlabel("Quantile (\u03c4)", fontsize=12, fontweight="bold")
ax2.set_ylabel("Coefficient Value", fontsize=12, fontweight="bold")
ax2.set_title("Panel B: Coefficient Paths (Unconstrained)", fontsize=14, fontweight="bold")
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Red curves in Panel A show observations where quantile predictions cross.")
print(
    "Panel B shows how coefficients evolve across quantiles (non-monotone paths can cause crossing)."
)

---

### 3.4 Method 1: Rearrangement

The rearrangement method (Chernozhukov et al. 2010) simply **sorts** the predicted quantiles for each observation. It is the simplest and fastest fix.

PanelBox implements this via `QuantileMonotonicity.rearrangement()`.

In [ ]:
# Apply rearrangement using PanelBox
rearranged_results = QuantileMonotonicity.rearrangement(results_for_crossing, X)

# Get rearranged predictions for the representative observation
preds_rearranged = {}
for tau in tau_grid:
    preds_rearranged[tau] = float(x_rep @ rearranged_results[tau].params)

# Show before vs after
print("REARRANGEMENT RESULTS")
print("=" * 60)
print(f"{'tau':>6s}  {'Before':>10s}  {'After':>10s}  {'Delta':>10s}")
print("-" * 42)
for tau in tau_grid:
    before = preds_unconstrained[tau]
    after = preds_rearranged[tau]
    diff = after - before
    print(f"{tau:6.1f}  {before:10.4f}  {after:10.4f}  {diff:+10.4f}")

# Verify monotonicity
rearr_vals = [preds_rearranged[tau] for tau in tau_grid]
is_monotone = all(rearr_vals[i] <= rearr_vals[i + 1] + 1e-10 for i in range(len(rearr_vals) - 1))
print(f"\nMonotonicity after rearrangement: {'Yes' if is_monotone else 'No'}")

### 3.5 Method 2: Isotonic Regression

Isotonic regression applies the Pool-Adjacent-Violators algorithm to the coefficient paths to enforce monotonicity. We use `QuantileMonotonicity.isotonic_regression()` on the coefficient matrix.

In [ ]:
# Build coefficient matrix (n_tau x n_coef)
coef_matrix = np.array([results_for_crossing[tau].params for tau in tau_grid])
print(f"Coefficient matrix shape: {coef_matrix.shape}  (n_tau x n_coef)")

# Apply isotonic regression to coefficient paths
monotone_coefs = QuantileMonotonicity.isotonic_regression(coef_matrix, tau_grid)

# Get isotonic predictions for the representative observation
preds_isotonic = {}
for i, tau in enumerate(tau_grid):
    preds_isotonic[tau] = float(x_rep @ monotone_coefs[i])

# Show results
print("\nISOTONIC REGRESSION RESULTS")
print("=" * 60)
print(f"{'tau':>6s}  {'Before':>10s}  {'After':>10s}  {'Delta':>10s}")
print("-" * 42)
for tau in tau_grid:
    before = preds_unconstrained[tau]
    after = preds_isotonic[tau]
    diff = after - before
    print(f"{tau:6.1f}  {before:10.4f}  {after:10.4f}  {diff:+10.4f}")

# Verify monotonicity
iso_vals = [preds_isotonic[tau] for tau in tau_grid]
is_monotone = all(iso_vals[i] <= iso_vals[i + 1] + 1e-10 for i in range(len(iso_vals) - 1))
print(f"\nMonotonicity after isotonic regression: {'Yes' if is_monotone else 'No'}")

### 3.6 Method 3: Constrained Quantile Regression

Constrained QR solves the quantile regression problem **jointly** across all quantiles, enforcing monotonicity constraints directly during estimation. This is the most principled approach but also the most computationally expensive.

PanelBox provides two methods:
- `constrained_qr()`: Hard constraints via `scipy.optimize.minimize` with trust-constr
- `simultaneous_qr()`: Soft penalty with $\lambda$ controlling the non-crossing penalty

In [ ]:
# Method 3a: Constrained QR (hard constraints)
# Note: Constrained QR is computationally expensive because it creates
# n_obs * (n_tau - 1) inequality constraints. We use a subsample for speed.
print("Estimating constrained quantile regression...")
print("(Using subsample of 100 observations for computational feasibility)")
t0 = time.time()

try:
    # Use a random subsample for constrained QR (full sample is too expensive)
    np.random.seed(42)
    n_sub = min(100, len(y))
    sub_idx = np.random.choice(len(y), n_sub, replace=False)
    X_sub = X[sub_idx]
    y_sub = y[sub_idx]

    constrained_betas = QuantileMonotonicity.constrained_qr(
        X_sub, y_sub, tau_grid, method="SLSQP", max_iter=200, verbose=False
    )
    t_constrained = time.time() - t0
    print(f"Constrained QR completed in {t_constrained:.2f} seconds.")

    # Get predictions for the representative observation (using full X)
    preds_constrained = {}
    for tau in tau_grid:
        preds_constrained[tau] = float(x_rep @ constrained_betas[tau])

    print("\nCONSTRAINED QR RESULTS")
    print("=" * 60)
    print(f"{'tau':>6s}  {'Before':>10s}  {'After':>10s}  {'Delta':>10s}")
    print("-" * 42)
    for tau in tau_grid:
        before = preds_unconstrained[tau]
        after = preds_constrained[tau]
        diff = after - before
        print(f"{tau:6.1f}  {before:10.4f}  {after:10.4f}  {diff:+10.4f}")

    # Verify monotonicity
    constr_vals = [preds_constrained[tau] for tau in tau_grid]
    is_monotone = all(
        constr_vals[i] <= constr_vals[i + 1] + 1e-10 for i in range(len(constr_vals) - 1)
    )
    print(f"\nMonotonicity after constrained QR: {'Yes' if is_monotone else 'No'}")

except Exception as e:
    print(f"Constrained QR failed: {e}")
    print("Falling back to simultaneous QR with soft penalty...")
    preds_constrained = None

In [ ]:
# Method 3b: Simultaneous QR with soft penalty (as alternative/complement)
print("Estimating simultaneous QR with non-crossing penalty...")
t0 = time.time()

try:
    simult_betas = QuantileMonotonicity.simultaneous_qr(
        X, y, tau_grid, lambda_nc=5.0, max_iter=200, verbose=True
    )
    t_simult = time.time() - t0
    print(f"Simultaneous QR completed in {t_simult:.2f} seconds.")

    # Get predictions
    preds_simultaneous = {}
    for tau in tau_grid:
        preds_simultaneous[tau] = float(x_rep @ simult_betas[tau])

    # Verify monotonicity
    simult_vals = [preds_simultaneous[tau] for tau in tau_grid]
    is_monotone = all(
        simult_vals[i] <= simult_vals[i + 1] + 1e-10 for i in range(len(simult_vals) - 1)
    )
    print(f"Monotonicity after simultaneous QR: {'Yes' if is_monotone else 'No'}")

except Exception as e:
    print(f"Simultaneous QR failed: {e}")
    preds_simultaneous = None

### 3.7 Comparison of All Methods

Now we compare all monotonicity enforcement methods side-by-side.

In [ ]:
# Collect all methods that succeeded
methods = {
    "Unconstrained": preds_unconstrained,
    "Rearrangement": preds_rearranged,
    "Isotonic": preds_isotonic,
}

if preds_constrained is not None:
    methods["Constrained"] = preds_constrained
if preds_simultaneous is not None:
    methods["Simultaneous"] = preds_simultaneous

# Create comparison plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

colors = ["steelblue", "#009E73", "#E69F00", "#D55E00", "#CC79A7"]
markers = ["o", "s", "^", "D", "v"]

# Panel A: Predicted quantiles
for (name, preds), color, marker in zip(methods.items(), colors, markers):
    vals = [preds[tau] for tau in tau_grid]
    ax1.plot(
        tau_grid,
        vals,
        marker=marker,
        linestyle="-",
        linewidth=2,
        markersize=7,
        label=name,
        color=color,
        alpha=0.8,
    )

ax1.set_xlabel("Quantile (\u03c4)", fontsize=12, fontweight="bold")
ax1.set_ylabel("Predicted Value", fontsize=12, fontweight="bold")
ax1.set_title("Panel A: Comparison of Monotonicity Methods", fontsize=14, fontweight="bold")
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# Panel B: Difference from unconstrained
for (name, preds), color, marker in list(zip(methods.items(), colors, markers))[1:]:
    diffs = [preds[tau] - preds_unconstrained[tau] for tau in tau_grid]
    ax2.plot(
        tau_grid,
        diffs,
        marker=marker,
        linestyle="-",
        linewidth=2,
        markersize=7,
        label=name,
        color=color,
        alpha=0.8,
    )

ax2.axhline(0, color="black", linestyle="--", linewidth=1.5, alpha=0.5)
ax2.set_xlabel("Quantile (\u03c4)", fontsize=12, fontweight="bold")
ax2.set_ylabel("Difference from Unconstrained", fontsize=12, fontweight="bold")
ax2.set_title("Panel B: Adjustments Made by Each Method", fontsize=14, fontweight="bold")
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "08_monotonicity_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Plot saved to {PLOTS_DIR / '08_monotonicity_comparison.png'}")

*Figure: Panel A shows predicted quantile values from each method. The unconstrained curve (blue) may exhibit non-monotonicity (crossing). The corrected methods (rearrangement, isotonic, constrained) produce monotone curves. Panel B shows how much each method adjusts predictions relative to the unconstrained baseline.*

---

## 4. Trade-off Analysis

Each monotonicity correction method trades off between **fidelity to the original estimates** and **monotonicity enforcement**. Let us quantify this.

In [ ]:
# Check loss function
def check_loss(u, tau):
    """Check (pinball) loss."""
    return u * (tau - (u < 0).astype(float))


# Compute total check loss over full sample for each method
print("TRADE-OFF ANALYSIS")
print("=" * 80)
print(
    f"{'Method':20s} {'Total Check Loss':>16s} {'Monotone?':>12s} {'Avg |Adj|':>12s} {'Max |Adj|':>12s}"
)
print("-" * 80)

# Build prediction matrices for each method
for name, preds_dict in methods.items():
    # Total check loss
    # For rearrangement/isotonic we use the fitted coefficients over the full sample
    total_loss = 0
    if name == "Unconstrained":
        for tau in tau_grid:
            pred = X @ results_for_crossing[tau].params
            resid = y - pred
            total_loss += np.sum(check_loss(resid, tau))
    elif name == "Rearrangement":
        for tau in tau_grid:
            pred = X @ rearranged_results[tau].params
            resid = y - pred
            total_loss += np.sum(check_loss(resid, tau))
    elif name == "Isotonic":
        for i, tau in enumerate(tau_grid):
            pred = X @ monotone_coefs[i]
            resid = y - pred
            total_loss += np.sum(check_loss(resid, tau))
    elif name == "Constrained" and preds_constrained is not None:
        for tau in tau_grid:
            pred = X @ constrained_betas[tau]
            resid = y - pred
            total_loss += np.sum(check_loss(resid, tau))
    elif name == "Simultaneous" and preds_simultaneous is not None:
        for tau in tau_grid:
            pred = X @ simult_betas[tau]
            resid = y - pred
            total_loss += np.sum(check_loss(resid, tau))

    # Check monotonicity for this observation
    vals = [preds_dict[tau] for tau in tau_grid]
    is_mono = all(vals[i] <= vals[i + 1] + 1e-10 for i in range(len(vals) - 1))
    mono_str = "Yes" if is_mono else "No"

    # Average adjustment from unconstrained
    if name != "Unconstrained":
        adjs = [abs(preds_dict[tau] - preds_unconstrained[tau]) for tau in tau_grid]
        avg_adj = np.mean(adjs)
        max_adj = np.max(adjs)
    else:
        avg_adj = 0.0
        max_adj = 0.0

    print(f"{name:20s} {total_loss:16.2f} {mono_str:>12s} {avg_adj:12.4f} {max_adj:12.4f}")

In [ ]:
# Systematic comparison across methods
# Note: MonotonicityComparison.compare_methods() has a known import bug (see correcoes/).
# We implement the comparison manually here.

print("SYSTEMATIC METHOD COMPARISON")
print("=" * 80)


# Compute total check loss for each method over full sample
def compute_total_check_loss(X, y, params_dict, tau_grid):
    """Compute total check loss across all quantiles."""
    total = 0
    for tau in tau_grid:
        pred = X @ params_dict[tau]
        resid = y - pred
        total += np.sum(resid * (tau - (resid < 0).astype(float)))
    return total


# Check crossing for each method
def check_crossing_full(X, params_dict, tau_grid):
    """Count crossing violations across all observations."""
    n_violations = 0
    for i in range(len(tau_grid) - 1):
        pred1 = X @ params_dict[tau_grid[i]]
        pred2 = X @ params_dict[tau_grid[i + 1]]
        n_violations += np.sum(pred1 > pred2 + 1e-10)
    return n_violations


# Prepare params dicts
method_params = {
    "Unconstrained": {tau: results_for_crossing[tau].params for tau in tau_grid},
    "Rearrangement": {tau: rearranged_results[tau].params for tau in tau_grid},
    "Isotonic": {tau: monotone_coefs[i] for i, tau in enumerate(tau_grid)},
}
if preds_constrained is not None:
    method_params["Constrained"] = {tau: constrained_betas[tau] for tau in tau_grid}
if preds_simultaneous is not None:
    method_params["Simultaneous"] = {tau: simult_betas[tau] for tau in tau_grid}

comparison_rows = []
for name, params_dict in method_params.items():
    loss = compute_total_check_loss(X, y, params_dict, tau_grid)
    violations = check_crossing_full(X, params_dict, tau_grid)
    n_total = len(X) * (len(tau_grid) - 1)
    pct_violations = 100 * violations / n_total

    comparison_rows.append(
        {
            "Method": name,
            "Total Check Loss": f"{loss:.2f}",
            "Violations": violations,
            "Pct Affected": f"{pct_violations:.2f}%",
            "Has Crossing": "Yes" if violations > 0 else "No",
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
print("Comparison Results:")
display(comparison_df)

> **Trade-off Insights**
>
> **Rearrangement**: Minimal change to predictions; fast (post-estimation); can create flat spots ("plateaus").
>
> **Isotonic**: Smooth monotone curve; more aggressive adjustment; can oversmooth.
>
> **Constrained**: Globally optimal subject to constraints; theoretically principled; computationally intensive; may sacrifice fit for monotonicity.
>
> **Recommendation**:
> - First try: Rearrangement (minimal intervention)
> - If heavy crossing: Constrained or Location-Scale (Notebook 05)
> - Never report unconstrained crossed quantiles in published work

In [ ]:
# Plot coefficient paths for each method
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_path = ["steelblue", "#009E73", "#E69F00", "#D55E00", "#CC79A7"]

for v_idx, vname in enumerate(["x1", "x2"]):
    for (method_name, params_dict), color in zip(method_params.items(), colors_path):
        coefs = [params_dict[tau][v_idx + 1] for tau in tau_grid]
        axes[v_idx].plot(
            tau_grid, coefs, "o-", linewidth=2, markersize=5, label=method_name, color=color
        )
    axes[v_idx].set_xlabel("Quantile (\u03c4)", fontsize=12)
    axes[v_idx].set_ylabel(f"Coefficient ({vname})", fontsize=12)
    axes[v_idx].set_title(f"{vname} Coefficient Path by Method", fontsize=14, fontweight="bold")
    axes[v_idx].legend(fontsize=9)
    axes[v_idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Coefficient paths show how each method adjusts the coefficient estimates.")

---

## 5. When to Accept/Fix Crossing

Not all crossing is equally problematic. Here is a decision guide for practitioners:

```
CROSSING DETECTED:
  |
  +-- Minor crossing (<5% of observations)?
  |   +-- YES -> Apply rearrangement, report both
  |
  +-- Tail quantiles only (tau < 0.1 or tau > 0.9)?
  |   +-- YES -> Consider dropping extreme quantiles OR
  |              use subsampling/bootstrap
  |
  +-- Systematic crossing (many quantiles)?
  |   +-- YES -> Model misspecification likely
  |        -> Try: Location-scale model (Notebook 05)
  |             OR add covariates/interactions
  |
  +-- Crossing in critical region of interest?
      +-- YES -> Use constrained estimation
```

In [ ]:
# Implement the decision guide as a diagnostic function
def diagnose_crossing(crossing_report, tau_list):
    """Provide recommendations based on crossing severity."""
    print("CROSSING DIAGNOSTIC")
    print("=" * 60)

    if not crossing_report.has_crossing:
        print("No crossing detected. No correction needed.")
        return

    print(f"Total inversions: {crossing_report.total_inversions}")
    print(f"Percentage affected: {crossing_report.pct_affected:.2f}%")

    # Classify severity
    if crossing_report.pct_affected < 5:
        severity = "MINOR"
        recommendation = "Apply rearrangement. Report both original and corrected."
    elif crossing_report.pct_affected < 20:
        severity = "MODERATE"
        recommendation = (
            "Consider constrained estimation or location-scale model. "
            "Check for model misspecification."
        )
    else:
        severity = "SEVERE"
        recommendation = (
            "Model misspecification likely. "
            "Re-evaluate model specification. "
            "Consider location-scale model (Notebook 05)."
        )

    # Check if tail-only
    tail_only = True
    for crossing in crossing_report.crossings:
        tau1, tau2 = crossing["tau_pair"]
        if tau1 >= 0.1 and tau2 <= 0.9:
            tail_only = False
            break

    print(f"\nSeverity: {severity}")
    if tail_only:
        print("Location: Tail quantiles only")
        print("  -> Consider dropping extreme quantiles or using bootstrap.")
    else:
        print("Location: Interior quantiles affected")

    print(f"\nRecommendation: {recommendation}")


# Run diagnostic
diagnose_crossing(crossing_report, tau_grid)

---

## Summary and Key Takeaways

1. **Quantile Crossing**: When estimated quantile curves cross ($\hat{Q}_{\tau_1}(Y|X) > \hat{Q}_{\tau_2}(Y|X)$ for $\tau_1 < \tau_2$), it violates the fundamental definition of quantiles. This produces uninterpretable predictions and negative implied densities.

2. **Rearrangement** (Chernozhukov et al. 2010): The simplest fix — sort predictions for each observation. Fast, minimal intervention, but can create plateaus.

3. **Isotonic Regression**: Smooths coefficient paths using the Pool-Adjacent-Violators algorithm. More aggressive but produces smooth monotone curves.

4. **Constrained QR**: Enforces monotonicity during estimation. Most principled but computationally expensive.

5. **Location-Scale Models** (Notebook 05): Prevent crossing by design through the parametric structure.

6. **Practical Rule**: Always check for crossing. For minor crossing, use rearrangement. For systematic crossing, reconsider the model.

### References

1. Chernozhukov, V., Fernández-Val, I., & Galichon, A. (2010). Quantile and Probability Curves Without Crossing. *Econometrica*, 78(3), 1093-1125.
2. Bondell, H. D., Reich, B. J., & Wang, H. (2010). Noncrossing Quantile Regression Curve Estimation. *Biometrika*, 97(4), 825-838.
3. Dette, H., & Volgushev, S. (2008). Non-Crossing Non-Parametric Estimates of Quantile Curves. *JRSSB*, 70(3), 609-627.

### Next Steps

- In **Notebook 05**, the Location-Scale model provides a natural framework that prevents crossing by design.
- Consider combining monotonicity checks with the bootstrap inference from **Notebook 07**.

---

## Exercises

### Exercise 1: Manual Rearrangement (Easy)

**Task**: Implement the rearrangement method manually (without using `QuantileMonotonicity.rearrangement()`).

**Steps**:
1. For each observation, collect predictions across all quantiles
2. Sort them to enforce monotonicity
3. Verify that your result matches the PanelBox implementation

**Hint**: The rearrangement is just sorting — `np.sort(predictions_for_observation_i)`.

In [ ]:
# Exercise 1: Implement rearrangement manually

# TODO: Get predictions for all observations and all quantiles
# predictions shape: (n_obs, n_tau)

# TODO: For each observation, sort the predictions

# TODO: Compare your result with QuantileMonotonicity.rearrangement()


### Exercise 2: Crossing from Small Samples vs Misspecification (Easy)

**Task**: Simulate two scenarios where crossing occurs:

a) **Small sample size**: Use the correct linear model but with very few observations (n=30).  
b) **Model misspecification**: Use a large sample but fit a linear QR to data generated from a nonlinear model.

Show that rearrangement fixes (a) but not (b) — i.e., in case (b), the crossing reflects a fundamental modeling problem.

**Hint**: For (b), generate $Y = \beta_0 + \beta_1 X + \beta_2 X^2 + \epsilon$ but fit only $Y \sim X$ (omitting $X^2$).

In [ ]:
# Exercise 2: Crossing from small samples vs misspecification
np.random.seed(42)

# TODO: Scenario (a): Small sample, correct model
# Generate data with n=30, fit QR for multiple quantiles, check crossing

# TODO: Scenario (b): Large sample, wrong model
# Generate nonlinear data, fit linear QR, check crossing

# TODO: Apply rearrangement to both and compare

### Exercise 3: Computational Time Comparison (Medium)

**Task**: Compare the computational time of the three methods:

1. Rearrangement (post-estimation)
2. Isotonic regression (coefficient smoothing)
3. Constrained QR (during estimation)

**Steps**:
1. Time each method using `time.time()`
2. Include the time for initial unconstrained estimation where relevant
3. Create a bar chart comparing the times

**Hint**: Rearrangement and isotonic are applied *after* estimation, so their total time includes QR fitting + correction. Constrained QR does everything in one step.

In [ ]:
# Exercise 3: Computational time comparison

# TODO: Time unconstrained QR estimation

# TODO: Time rearrangement (post-estimation correction only)

# TODO: Time isotonic regression (post-estimation correction only)

# TODO: Time constrained QR (full estimation)

# TODO: Create bar chart comparing times


### Exercise 4: Application to Real Data (Medium)

**Task**: Apply all three monotonicity methods to the Card education data from Notebook 01.

**Steps**:
1. Load `card_education.csv`
2. Estimate QR for $\tau \in \{0.05, 0.1, 0.15, ..., 0.95\}$ using `lwage ~ educ + exper + exper^2`
3. Check for crossing
4. Apply all three methods
5. Which gives the most plausible results? Why?

In [ ]:
# Exercise 4: Apply methods to real data

# TODO: Load card_education.csv

# TODO: Prepare design matrix (intercept, educ, exper, exper_sq)

# TODO: Estimate QR for fine grid of quantiles

# TODO: Detect crossing

# TODO: Apply rearrangement, isotonic, constrained

# TODO: Compare results and discuss


### Exercise 5: Location-Scale and Crossing (Hard)

**Task**: Derive conditions under which a location-scale model NEVER has crossing.

**Background**: The location-scale model specifies:

$$Q_\tau(Y|X) = X'\beta + (X'\gamma) \cdot F^{-1}(\tau)$$

where $F$ is a distribution function.

**Questions**:
1. Under what condition on $X'\gamma$ is crossing impossible?
2. Verify numerically by estimating the location-scale model from Notebook 05
3. What happens when the condition is violated?

In [ ]:
# Exercise 5: Location-scale and crossing

# TODO: Write the mathematical derivation as comments

# TODO: Numerically verify with simulated data
# Hint: Location-scale model prevents crossing when X'gamma > 0 for all X,
#       because F^{-1}(tau) is increasing in tau.


### Exercise 6: Bootstrap with Rearrangement (Hard)

**Task**: Incorporate monotonicity correction into a bootstrap procedure.

**Steps**:
1. In each bootstrap replication:
   a. Resample (with replacement, by cluster)
   b. Estimate QR for all quantiles
   c. Apply rearrangement
   d. Store corrected coefficients
2. Compute bootstrap standard errors and confidence intervals
3. Compare with naive bootstrap (without rearrangement)

**Discussion**: Does correcting within the bootstrap change inference?

In [ ]:
# Exercise 6: Bootstrap with rearrangement
np.random.seed(42)

# TODO: Implement bootstrap with and without rearrangement
# n_boot = 100 (keep small for speed)

# TODO: Compare bootstrap SEs with and without correction

# TODO: Discuss the impact on inference